# Product Review Sentiment Analysis — Implementation and Results

**Team 15** · 23BCT0194 · 24BAI0050 · 22BCE2689

This notebook builds and tests the proposed system, then saves every table and graph needed for **Section 4 (Results and Implementation)** and **Section 5 (Performance and Metrics)** of the report.

| Part | Report section | What it produces |
|---|---|---|
| Module 1 – Data input and preprocessing | 4.2 | cleaning steps table, class distribution, review length chart |
| Baseline – base paper's TF-IDF models | 5 | accuracy / precision / recall / F1 of NB, Linear SVC, LR, MLP, RF |
| Module 2 – Multi-task transformer | 4.3 | sentiment and emotion results, confusion matrices, training curves, ablation |
| Module 3 – Aspect extraction and polarity | 4.4 | aspect polarity accuracy, top aspects chart, example table |
| Module 4 – Explanation and output | 4.5 | key-word explanations, hard-case test, result card, bulk-file speed, dashboard |
| Comparison | 5 | existing vs proposed table and graphs |

## How to run
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough).
2. **Runtime → Run all.** It takes about 1–2 hours.
3. At the end, `results.zip` downloads automatically. Send that file back so the report can be filled in.

*Optional:* to run on the base paper's dataset instead, upload Kaggle's **Amazon Fine Food Reviews** `Reviews.csv` to `/content/` and set `DATA_SOURCE = "finefood"` below.

In [ ]:
# 0. Install packages (about 1–2 minutes)
!pip -q install -U "transformers>=4.44" "datasets>=2.20" accelerate captum
!pip -q install spacy nltk
!python -m spacy download en_core_web_sm -q

In [ ]:
# 0. Settings — change here if needed
DATA_SOURCE   = "amazon2023"                  # "amazon2023" or "finefood"
CATEGORY      = "Cell_Phones_and_Accessories" # Amazon Reviews 2023 category
MAX_PER_CLASS = 40000      # max reviews kept per sentiment class (Negative / Neutral / Positive)
SCAN_LIMIT    = 1_500_000  # max reviews read while sampling
FINEFOOD_CSV  = "/content/Reviews.csv"

MODEL_NAME    = "distilbert-base-uncased"
MAX_LEN       = 128
BATCH_SIZE    = 32
EPOCHS        = 2
LR            = 2e-5
LAMBDA_EMO    = 0.5        # weight of the emotion loss
MU_ASP        = 0.5        # weight of the aspect loss
ASPECT_EVERY  = 16         # add one aspect batch every N steps
RUN_ABLATION  = True       # also train a sentiment-only model for comparison
CHUNK_SIZE    = 50000      # rows per chunk in bulk-file mode
SEED          = 42
OUT           = "results"

SEMEVAL_TRAIN_XML = "/content/Laptop_Train_v2.xml"     # only used if the Hugging Face copy fails
SEMEVAL_TEST_XML  = "/content/Laptops_Test_Gold.xml"

In [ ]:
# 0. Imports, plotting style and helpers
import os, re, io, html, json, time, math, random, copy, zipfile, warnings
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEV.type == "cuda"
print("Device:", DEV, torch.cuda.get_device_name(0) if USE_AMP else "")

SENT_LABELS = ["Negative", "Neutral", "Positive"]
EMO_LABELS  = ["Joy", "Sadness", "Anger", "Fear", "Disgust", "Surprise"]

# colours: diverging red-gray-blue for sentiment, blue = proposed, gray = existing
BLUE, ORANGE, GRAY, RED = "#2a78d6", "#eb6834", "#8a8986", "#e34948"
SENT_COLORS = [RED, GRAY, BLUE]
SEQ = LinearSegmentedColormap.from_list("seq", ["#f4f8fd", "#cde2fb", "#86b6ef", "#2a78d6", "#184f95", "#0d366b"])
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.edgecolor": "#b9b8b3", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.titlesize": 12, "axes.titleweight": "bold", "legend.frameon": False})

RESULTS = {}            # every number the report needs ends up here

def save_fig(fig, name):
    fig.tight_layout(); fig.savefig(f"{OUT}/{name}.png", bbox_inches="tight", facecolor="white"); plt.show()

def save_table(df, name):
    df.to_csv(f"{OUT}/{name}.csv", index=False); display(df)

def bar_labels(ax, bars, fmt="{:.3f}"):
    for b in bars:
        ax.annotate(fmt.format(b.get_height()), (b.get_x() + b.get_width() / 2, b.get_height()),
                    ha="center", va="bottom", fontsize=8, color="#52514e",
                    xytext=(0, 2), textcoords="offset points")

def scores(y, p, labels):
    acc = accuracy_score(y, p)
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    _, _, f1c, _ = precision_recall_fscore_support(y, p, labels=list(range(len(labels))), zero_division=0)
    return {"accuracy": acc, "precision": pr, "recall": rc, "macro_f1": f1,
            "per_class_f1": {l: float(v) for l, v in zip(labels, f1c)}}

def plot_cm(ax, y, p, labels, title):
    cm = confusion_matrix(y, p, labels=list(range(len(labels))), normalize="true")
    ax.imshow(cm, cmap=SEQ, vmin=0, vmax=1); ax.grid(False)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if cm[i, j] > 0.55 else "#0b0b0b")
    ax.set_xticks(range(len(labels)), labels, rotation=30 if len(labels) > 3 else 0)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)

## Module 1 — Data Input and Preprocessing  *(report Section 4.2)*
Reads reviews in a stream, keeps a balanced sample, removes empty rows, duplicates, HTML, links and very short reviews, and maps star ratings to sentiment labels. **Stop-words like "not" and "but" are kept**, because the transformer needs them.

In [ ]:
def clean_text(t):
    t = html.unescape(str(t))
    t = re.sub(r"<[^>]+>", " ", t)                 # HTML tags
    t = re.sub(r"http\S+|www\.\S+", " ", t)        # links
    return re.sub(r"\s+", " ", t).strip()

def rating_to_label(r):
    r = float(r)
    return 0 if r <= 2 else (1 if r == 3 else 2)

def load_amazon_2023(category, max_per_class, scan_limit):
    from datasets import load_dataset
    url = f"hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/{category}.jsonl"
    try:
        stream = load_dataset("json", data_files=url, split="train", streaming=True)
    except Exception as e:
        print("Direct file read failed, trying the dataset script:", str(e)[:200])
        stream = load_dataset("McAuley-Lab/Amazon-Reviews-2023", f"raw_review_{category}",
                              split="full", streaming=True, trust_remote_code=True)
    counts, rows, scanned = [0, 0, 0], [], 0
    for ex in tqdm(stream, desc="Reading reviews", total=scan_limit):
        scanned += 1
        if scanned > scan_limit or min(counts) >= max_per_class:
            break
        lab = rating_to_label(ex["rating"])
        if counts[lab] >= max_per_class:
            continue
        rows.append({"review_text": ex.get("text"), "rating": ex.get("rating"),
                     "product_id": ex.get("parent_asin") or ex.get("asin"),
                     "timestamp": ex.get("timestamp")})
        counts[lab] += 1
    print(f"Scanned {scanned:,} reviews, kept {len(rows):,}  (per class: {counts})")
    return pd.DataFrame(rows), scanned

def load_finefood(path, max_per_class):
    df = pd.read_csv(path, usecols=["ProductId", "Score", "Time", "Text"])
    df = df.rename(columns={"Text": "review_text", "Score": "rating",
                            "ProductId": "product_id", "Time": "timestamp"})
    df["timestamp"] = df["timestamp"] * 1000
    scanned = len(df)
    df["_l"] = df["rating"].map(rating_to_label)
    df = df.sample(frac=1, random_state=SEED).groupby("_l").head(max_per_class).drop(columns="_l")
    return df.reset_index(drop=True), scanned

def preprocess(df):
    steps = [("Rows read", len(df))]
    df = df.dropna(subset=["review_text", "rating"]).copy()
    steps.append(("After removing empty rows", len(df)))
    df["review_text"] = df["review_text"].astype(str)
    df = df.drop_duplicates(subset="review_text")
    steps.append(("After removing duplicate reviews", len(df)))
    df["clean_text"] = df["review_text"].map(clean_text)
    df = df[df["clean_text"].str.split().str.len() >= 3]
    steps.append(("After removing reviews under 3 words", len(df)))
    df["label"] = df["rating"].map(rating_to_label)
    return df.reset_index(drop=True), pd.DataFrame(steps, columns=["Step", "Rows"])

In [ ]:
t0 = time.time()
if DATA_SOURCE == "finefood":
    raw_df, scanned = load_finefood(FINEFOOD_CSV, MAX_PER_CLASS)
    DATA_NAME = "Amazon Fine Food Reviews"
else:
    raw_df, scanned = load_amazon_2023(CATEGORY, MAX_PER_CLASS, SCAN_LIMIT)
    DATA_NAME = f"Amazon Reviews 2023 ({CATEGORY.replace('_', ' ')})"
t_load = time.time() - t0

t0 = time.time()
df, steps_df = preprocess(raw_df)
t_clean = time.time() - t0
print(f"Loaded in {t_load:.0f}s, cleaned in {t_clean:.1f}s ({len(raw_df) / max(t_clean, 1e-6):,.0f} reviews/s)")
save_table(steps_df, "m1_preprocessing_steps")

train_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df  = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df["label"], random_state=SEED)
split_df = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Reviews": [len(train_df), len(val_df), len(test_df)],
    **{l: [int((d["label"] == i).sum()) for d in (train_df, val_df, test_df)] for i, l in enumerate(SENT_LABELS)}})
save_table(split_df, "m1_splits")

lens = df["clean_text"].str.split().str.len()
RESULTS["data"] = {"source": DATA_NAME, "scanned": int(scanned), "kept_after_cleaning": len(df),
                   "train": len(train_df), "val": len(val_df), "test": len(test_df),
                   "load_seconds": round(t_load, 1), "clean_seconds": round(t_clean, 2),
                   "clean_reviews_per_sec": round(len(raw_df) / max(t_clean, 1e-6)),
                   "median_words": float(lens.median()), "mean_words": round(float(lens.mean()), 1),
                   "pct_over_128_words": round(float((lens > 128).mean() * 100), 1),
                   "steps": steps_df.to_dict("records")}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
cnt = df["label"].value_counts().sort_index()
bars = axes[0].bar(SENT_LABELS, cnt.values, color=SENT_COLORS, width=0.6)
bar_labels(axes[0], bars, "{:,.0f}")
axes[0].set_title("Reviews per sentiment class"); axes[0].set_ylabel("Reviews")
axes[1].hist(lens.clip(upper=400), bins=np.arange(0, 410, 10), color=BLUE, edgecolor="white", linewidth=0.5)
axes[1].axvline(128, color="#52514e", ls="--", lw=1)
axes[1].text(132, axes[1].get_ylim()[1] * 0.9, "128 (max length)", fontsize=8, color="#52514e")
axes[1].set_title("Review length"); axes[1].set_xlabel("Words per review (clipped at 400)"); axes[1].set_ylabel("Reviews")
save_fig(fig, "m1_class_distribution_and_length")

## Baseline — the base paper's method  *(existing methodology, report Section 5)*
Same steps as the base paper: lowercase, remove stop-words (this also removes "not"), lemmatize, TF-IDF features, then Naive Bayes, Linear SVC, Logistic Regression, MLP and Random Forest. Grid search tunes NB and SVC. MLP and Random Forest are trained on a 30,000-review subset to keep the run time reasonable.

In [ ]:
import nltk
for p in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(p, quiet=True)
try:
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    SW, LEM = set(stopwords.words("english")), WordNetLemmatizer(); LEM.lemmatize("tests")
except LookupError:                                   # NLTK data unavailable -> scikit-learn stop-words, no lemmatizer
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    class _NoLem:
        def lemmatize(self, w): return w
    SW, LEM = set(ENGLISH_STOP_WORDS), _NoLem()
    print("NLTK data not available - using scikit-learn stop-words without lemmatization")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

def base_paper_preprocess(t):
    t = re.sub(r"[^a-z\s]", " ", t.lower())
    return " ".join(LEM.lemmatize(w) for w in t.split() if w not in SW and len(w) > 2)

t0 = time.time()
Xtr_txt = train_df["clean_text"].map(base_paper_preprocess)
Xte_txt = test_df["clean_text"].map(base_paper_preprocess)
vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
Xtr, Xte = vec.fit_transform(Xtr_txt), vec.transform(Xte_txt)
ytr, yte = train_df["label"].values, test_df["label"].values
print(f"TF-IDF features ready in {time.time() - t0:.0f}s, shape {Xtr.shape}")

sub = np.random.RandomState(SEED).choice(Xtr.shape[0], min(30000, Xtr.shape[0]), replace=False)
BASELINES = {
    "Naive Bayes":         (GridSearchCV(MultinomialNB(), {"alpha": [0.1, 0.5, 1.0]}, cv=3, scoring="f1_macro"), False),
    "Linear SVC":          (GridSearchCV(LinearSVC(), {"C": [0.1, 0.5, 1.0]}, cv=3, scoring="f1_macro"), False),
    "Logistic Regression": (LogisticRegression(max_iter=1000, C=1.0), False),
    "MLP":                 (MLPClassifier(hidden_layer_sizes=(128,), max_iter=15, early_stopping=True, random_state=SEED), True),
    "Random Forest":       (RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED), True),
}
BASE_RESULTS, BASE_PREDS, BASE_MODELS = {}, {}, {}
for name, (clf, use_sub) in BASELINES.items():
    t0 = time.time()
    clf.fit(Xtr[sub] if use_sub else Xtr, ytr[sub] if use_sub else ytr)
    tt = time.time() - t0
    p = clf.predict(Xte)
    BASE_RESULTS[name] = {**scores(yte, p, SENT_LABELS), "train_seconds": round(tt, 1),
                          "trained_on": int(len(sub) if use_sub else Xtr.shape[0])}
    BASE_PREDS[name], BASE_MODELS[name] = p, clf
    print(f"{name:20s} acc={BASE_RESULTS[name]['accuracy']:.4f}  macroF1={BASE_RESULTS[name]['macro_f1']:.4f}  ({tt:.0f}s)")

BEST_BASE = max(BASE_RESULTS, key=lambda k: BASE_RESULTS[k]["macro_f1"])
print("Best baseline:", BEST_BASE)
RESULTS["baselines"] = BASE_RESULTS; RESULTS["best_baseline"] = BEST_BASE

## Module 2 — Multi-Task Transformer for Sentiment and Emotion  *(report Section 4.3)*
One DistilBERT encoder with three output heads: sentiment (3 classes), emotion (6 Ekman classes, trained on GoEmotions) and aspect polarity (trained on SemEval-2014 laptops, used in Module 3). Training alternates between the datasets: every step uses one review batch, one emotion batch, and every `ASPECT_EVERY` steps one aspect batch.

Loss: **L = L_sentiment + λ·L_emotion + μ·L_aspect**

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# --- GoEmotions -> Ekman's six emotions (official GoEmotions mapping)
EKMAN = {"Joy": ["joy", "amusement", "approval", "excitement", "gratitude", "love", "optimism",
                 "relief", "pride", "admiration", "desire", "caring"],
         "Sadness": ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
         "Anger": ["anger", "annoyance", "disapproval"],
         "Fear": ["fear", "nervousness"],
         "Disgust": ["disgust"],
         "Surprise": ["surprise", "realization", "confusion", "curiosity"]}
FINE2EK = {f: EMO_LABELS.index(k) for k, v in EKMAN.items() for f in v}

def load_goemotions():
    ds = load_dataset("google-research-datasets/go_emotions", "simplified")
    try:
        names = ds["train"].features["labels"].feature.names
    except AttributeError:   # official GoEmotions label order
        names = ["admiration", "amusement", "anger", "annoyance", "approval", "caring", "confusion", "curiosity",
                 "desire", "disappointment", "disapproval", "disgust", "embarrassment", "excitement", "fear",
                 "gratitude", "grief", "joy", "love", "nervousness", "optimism", "pride", "realization", "relief",
                 "remorse", "sadness", "surprise", "neutral"]
    def conv(split):
        rows = []
        for t, labs in zip(split["text"], split["labels"]):
            ek = {FINE2EK[names[l]] for l in labs if names[l] in FINE2EK}
            if len(ek) == 1:                       # keep sentences with exactly one Ekman emotion
                rows.append((t, ek.pop()))
        return pd.DataFrame(rows, columns=["text", "label"])
    return conv(ds["train"]), conv(ds["validation"]), conv(ds["test"])

# --- SemEval-2014 laptops (aspect, polarity)
ASP_MAP = {"negative": 0, "neutral": 1, "positive": 2}
def load_semeval_xml(path):
    import xml.etree.ElementTree as ET
    rows = []
    for s in ET.parse(path).getroot().iter("sentence"):
        at = s.find("aspectTerms")
        if at is None: continue
        for a in at.findall("aspectTerm"):
            if a.get("polarity") in ASP_MAP:
                rows.append({"text": s.find("text").text, "aspect": a.get("term"), "label": ASP_MAP[a.get("polarity")]})
    return pd.DataFrame(rows)

def load_semeval():
    # 1) official gold XML files, if you uploaded them
    if os.path.exists(SEMEVAL_TRAIN_XML) and os.path.exists(SEMEVAL_TEST_XML):
        return load_semeval_xml(SEMEVAL_TRAIN_XML), load_semeval_xml(SEMEVAL_TEST_XML), "SemEval-2014 XML files"
    # 2) Hugging Face copy
    try:
        ds = load_dataset("tomaarsen/setfit-absa-semeval-laptops")
        def conv(split):
            d = split.to_pandas()
            d["label"] = d["label"].astype(str).str.lower().str.strip()
            d = d[d["label"].isin(list(ASP_MAP))]
            return pd.DataFrame({"text": d["text"].values, "aspect": d["span"].values,
                                 "label": d["label"].map(ASP_MAP).values})
        tr, te = conv(ds["train"]), conv(ds["test"])
        src = "tomaarsen/setfit-absa-semeval-laptops"
        if len(te) == 0:   # this copy's test split has no labels -> hold out 15% of the labelled training data
            tr, te = train_test_split(tr, test_size=0.15, stratify=tr["label"], random_state=SEED)
            src += " (labelled train split; 15% held out as test)"
            print("SemEval test split has no labels - using 15% of the labelled training data as the test set.")
        return tr.reset_index(drop=True), te.reset_index(drop=True), src
    except Exception as e:
        print("Hugging Face SemEval copy failed:", str(e)[:200])
    print("SemEval not available -> aspect head skipped; Module 3 will use the clause-based fallback.")
    return None, None, None

emo_tr, emo_va, emo_te = load_goemotions()
asp_tr_all, asp_te, ASP_SOURCE = load_semeval()
if asp_tr_all is not None:
    asp_tr, asp_va = train_test_split(asp_tr_all, test_size=0.1, stratify=asp_tr_all["label"], random_state=SEED)

aux_df = pd.DataFrame({
    "Dataset": ["GoEmotions (mapped to Ekman 6)"] * 3 + (["SemEval-2014 Laptops"] * 3 if asp_tr_all is not None else []),
    "Split": ["Train", "Validation", "Test"] * (2 if asp_tr_all is not None else 1),
    "Rows": [len(emo_tr), len(emo_va), len(emo_te)] + ([len(asp_tr), len(asp_va), len(asp_te)] if asp_tr_all is not None else [])})
save_table(aux_df, "m2_aux_datasets")
emo_dist = emo_tr["label"].value_counts().reindex(range(6), fill_value=0)
RESULTS["aux_data"] = {"goemotions": {"train": len(emo_tr), "val": len(emo_va), "test": len(emo_te),
                                      "train_class_counts": {EMO_LABELS[i]: int(c) for i, c in emo_dist.items()}},
                       "semeval": None if asp_tr_all is None else {"source": ASP_SOURCE, "train": len(asp_tr),
                                                                   "val": len(asp_va), "test": len(asp_te)}}

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDS(Dataset):
    def __init__(self, texts, labels, pairs=None):
        enc = tok(list(texts), text_pair=None if pairs is None else list(pairs), truncation=True,
                  padding="max_length", max_length=MAX_LEN, return_tensors="pt")
        self.ids, self.mask = enc["input_ids"], enc["attention_mask"]
        self.y = torch.tensor(np.asarray(labels), dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.y[i]

t0 = time.time()
sent_tr, sent_va, sent_te = (TextDS(d["clean_text"], d["label"]) for d in (train_df, val_df, test_df))
emo_tr_ds, emo_va_ds, emo_te_ds = (TextDS(d["text"], d["label"]) for d in (emo_tr, emo_va, emo_te))
if asp_tr_all is not None:
    asp_tr_ds, asp_va_ds, asp_te_ds = (TextDS(d["text"], d["label"], d["aspect"]) for d in (asp_tr, asp_va, asp_te))
else:
    asp_tr_ds = asp_va_ds = asp_te_ds = None
print(f"Tokenised everything in {time.time() - t0:.0f}s")

counts = np.bincount(emo_tr["label"], minlength=6).astype(float)
EMO_W = torch.tensor((counts.sum() / (6 * np.maximum(counts, 1))) ** 0.5, dtype=torch.float, device=DEV)

class MultiTaskModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self.drop = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({"sent": nn.Linear(h, 3), "emo": nn.Linear(h, 6), "asp": nn.Linear(h, 3)})
    def forward(self, input_ids, attention_mask, task="sent"):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.heads[task](self.drop(cls))

def cycle(loader):
    if loader is None: return None
    def gen():
        while True:
            for b in loader: yield b
    return gen()

@torch.no_grad()
def evaluate(model, ds, task, bs=128):
    model.eval(); Y, P, C = [], [], []
    for ids, mask, y in DataLoader(ds, batch_size=bs):
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
            pr = model(ids.to(DEV), mask.to(DEV), task).float().softmax(-1)
        Y.append(y); P.append(pr.argmax(-1).cpu()); C.append(pr.max(-1).values.cpu())
    return torch.cat(Y).numpy(), torch.cat(P).numpy(), torch.cat(C).numpy()

def train_model(lam_emo=LAMBDA_EMO, use_aspect=True, epochs=EPOCHS, tag="multi-task"):
    set_seed()
    model = MultiTaskModel(MODEL_NAME).to(DEV)
    sent_loader = DataLoader(sent_tr, batch_size=BATCH_SIZE, shuffle=True)
    emo_it = cycle(DataLoader(emo_tr_ds, batch_size=BATCH_SIZE, shuffle=True)) if lam_emo > 0 else None
    asp_it = cycle(DataLoader(asp_tr_ds, batch_size=BATCH_SIZE, shuffle=True)) if (use_aspect and asp_tr_ds is not None) else None
    total = epochs * len(sent_loader)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    hist = {"loss": [], "val_sent_f1": [], "val_emo_f1": [], "val_asp_f1": [], "epoch_seconds": []}
    best_f1, best_state, step = -1, None, 0
    for ep in range(epochs):
        model.train(); t0 = time.time()
        for ids, mask, y in tqdm(sent_loader, desc=f"{tag} · epoch {ep + 1}/{epochs}"):
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
                loss = F.cross_entropy(model(ids.to(DEV), mask.to(DEV), "sent"), y.to(DEV))
                if emo_it is not None:
                    e_ids, e_mask, e_y = next(emo_it)
                    loss = loss + lam_emo * F.cross_entropy(model(e_ids.to(DEV), e_mask.to(DEV), "emo"),
                                                            e_y.to(DEV), weight=EMO_W)
                if asp_it is not None and step % ASPECT_EVERY == 0:
                    a_ids, a_mask, a_y = next(asp_it)
                    loss = loss + MU_ASP * F.cross_entropy(model(a_ids.to(DEV), a_mask.to(DEV), "asp"), a_y.to(DEV))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            hist["loss"].append(float(loss.item())); step += 1
        hist["epoch_seconds"].append(round(time.time() - t0, 1))
        y, p, _ = evaluate(model, sent_va, "sent"); f1 = scores(y, p, SENT_LABELS)["macro_f1"]
        hist["val_sent_f1"].append(f1)
        if emo_it is not None:
            y, p, _ = evaluate(model, emo_va_ds, "emo"); hist["val_emo_f1"].append(scores(y, p, EMO_LABELS)["macro_f1"])
        if asp_it is not None:
            y, p, _ = evaluate(model, asp_va_ds, "asp"); hist["val_asp_f1"].append(scores(y, p, SENT_LABELS)["macro_f1"])
        print(f"epoch {ep + 1}: val sentiment macro-F1 = {f1:.4f}"
              + (f", emotion = {hist['val_emo_f1'][-1]:.4f}" if hist["val_emo_f1"] else "")
              + (f", aspect = {hist['val_asp_f1'][-1]:.4f}" if hist["val_asp_f1"] else ""))
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, hist

In [ ]:
model, hist = train_model()
ASPECT_HEAD = asp_tr_ds is not None

ys, ps, cs = evaluate(model, sent_te, "sent")
ye, pe, ce = evaluate(model, emo_te_ds, "emo")
PROP = {**scores(ys, ps, SENT_LABELS), "train_seconds": round(sum(hist["epoch_seconds"]), 1),
        "trained_on": len(train_df)}
EMO_RES = scores(ye, pe, EMO_LABELS)
RESULTS["proposed_sentiment"] = PROP
RESULTS["proposed_emotion"] = EMO_RES
RESULTS["training"] = {k: v for k, v in hist.items() if k != "loss"}
print("Proposed sentiment (test):", {k: round(v, 4) for k, v in PROP.items() if isinstance(v, float)})
print("Proposed emotion (GoEmotions test):", {k: round(v, 4) for k, v in EMO_RES.items() if isinstance(v, float)})
print(classification_report(ys, ps, labels=list(range(3)), target_names=SENT_LABELS, digits=4, zero_division=0))
print(classification_report(ye, pe, labels=list(range(6)), target_names=EMO_LABELS, digits=4, zero_division=0))
torch.save(model.state_dict(), "multitask_model.pt")

In [ ]:
# training curves (two separate charts, one axis each)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
loss = pd.Series(hist["loss"]).rolling(50, min_periods=1).mean()
axes[0].plot(loss.values, color=BLUE, lw=2)
axes[0].set_title("Training loss (moving average of 50 steps)"); axes[0].set_xlabel("Step"); axes[0].set_ylabel("Combined loss")
ep = np.arange(1, len(hist["val_sent_f1"]) + 1)
axes[1].plot(ep, hist["val_sent_f1"], "o-", color=BLUE, lw=2, ms=8, label="Sentiment")
if hist["val_emo_f1"]: axes[1].plot(ep, hist["val_emo_f1"], "o-", color=ORANGE, lw=2, ms=8, label="Emotion")
if hist["val_asp_f1"]: axes[1].plot(ep, hist["val_asp_f1"], "o-", color="#1baf7a", lw=2, ms=8, label="Aspect")
axes[1].set_xticks(ep); axes[1].set_ylim(0, 1); axes[1].legend(loc="lower right")
axes[1].set_title("Validation macro-F1 per epoch"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro-F1")
save_fig(fig, "m2_training_curves")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"width_ratios": [1, 1.4]})
plot_cm(axes[0], ys, ps, SENT_LABELS, "Sentiment (proposed)")
plot_cm(axes[1], ye, pe, EMO_LABELS, "Emotion (proposed)")
save_fig(fig, "m2_confusion_matrices")

fig, ax = plt.subplots(figsize=(7, 3.6))
bars = ax.bar(EMO_LABELS, [EMO_RES["per_class_f1"][l] for l in EMO_LABELS], color=BLUE, width=0.6)
bar_labels(ax, bars, "{:.2f}"); ax.set_ylim(0, 1.05)
ax.set_title("Emotion F1 per class (GoEmotions test)"); ax.set_ylabel("F1")
save_fig(fig, "m2_emotion_f1_per_class")

In [ ]:
# Ablation: does adding the emotion and aspect tasks help sentiment?
if RUN_ABLATION:
    single, hist_single = train_model(lam_emo=0, use_aspect=False, tag="sentiment-only")
    y1, p1, _ = evaluate(single, sent_te, "sent")
    ABL = {**scores(y1, p1, SENT_LABELS), "train_seconds": round(sum(hist_single["epoch_seconds"]), 1),
           "trained_on": len(train_df)}
    RESULTS["ablation_sentiment_only"] = ABL
    abl_df = pd.DataFrame([
        {"Model": "Sentiment-only DistilBERT", **{k: round(ABL[k], 4) for k in ["accuracy", "precision", "recall", "macro_f1"]}},
        {"Model": "Multi-task DistilBERT (proposed)", **{k: round(PROP[k], 4) for k in ["accuracy", "precision", "recall", "macro_f1"]}}])
    save_table(abl_df, "m2_ablation")
    del single; torch.cuda.empty_cache()

## Module 3 — Aspect Extraction and Aspect-Level Sentiment  *(report Section 4.4)*
Aspects are noun phrases that match a list of the most frequent product nouns. Each (review, aspect) pair goes to the aspect head as a sentence pair. If SemEval was not available, a fallback runs the sentiment head on the clause that contains the aspect.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner"])
GENERIC = {"product", "item", "thing", "one", "star", "stars", "review", "time", "lot", "way", "bit", "day",
           "days", "amazon", "use", "issue", "problem", "purchase", "order", "month", "week", "year", "people",
           "anything", "something", "nothing", "everything", "part", "piece", "kind", "reason", "fact", "life",
           "phone", "iphone", "samsung", "galaxy", "apple", "android", "work", "works", "fit"}
SYN = {"cam": "camera", "photo": "picture", "pic": "picture", "batt": "battery", "cost": "price",
       "money": "price", "value": "price", "shipping": "delivery", "shipment": "delivery",
       "audio": "sound", "mic": "microphone", "display": "screen", "cord": "cable", "wire": "cable"}

def norm_aspect(w):
    w = w.lower(); return SYN.get(w, w)

def chunk_nouns(nc):
    # every noun in the phrase, left to right: "camera quality" -> camera, quality
    return [norm_aspect(t.lemma_) for t in nc if t.pos_ in ("NOUN", "PROPN")]

def build_aspect_vocab(texts, top_k=25):
    cnt = Counter()
    for doc in nlp.pipe((t[:1000] for t in texts), batch_size=256):
        for nc in doc.noun_chunks:
            for w in chunk_nouns(nc):
                if w.isalpha() and len(w) > 2 and w not in GENERIC:
                    cnt[w] += 1
    return [w for w, _ in cnt.most_common(top_k)]

def extract_aspects(texts):
    out = []
    for doc in nlp.pipe((t[:1000] for t in texts), batch_size=256):
        found = []
        for nc in doc.noun_chunks:
            hit = next((w for w in chunk_nouns(nc) if w in ASPECT_VOCAB), None)   # first matching noun
            if hit and hit not in found:
                found.append(hit)
        out.append(found)
    return out

SPLIT = re.compile(r"\b(?:but|however|although|though|yet|whereas)\b|[;,.!?]", re.I)
def clause_for(text, aspect):
    for part in SPLIT.split(text):
        if aspect.lower() in part.lower(): return part.strip()
    return text

@torch.no_grad()
def predict_texts(model, texts, task="sent", pairs=None, bs=128):
    model.eval(); P, C = [], []
    texts = list(texts); pairs = None if pairs is None else list(pairs)
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], text_pair=None if pairs is None else pairs[i:i + bs], truncation=True,
                  padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEV)
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
            pr = model(enc["input_ids"], enc["attention_mask"], task).float().softmax(-1)
        P.append(pr.argmax(-1).cpu()); C.append(pr.max(-1).values.cpu())
    if not P: return np.array([], dtype=int), np.array([])
    return torch.cat(P).numpy(), torch.cat(C).numpy()

def aspect_polarity(texts, aspects):
    if ASPECT_HEAD:
        return predict_texts(model, texts, "asp", pairs=aspects)
    return predict_texts(model, [clause_for(t, a) for t, a in zip(texts, aspects)], "sent")

t0 = time.time()
ASPECT_VOCAB = build_aspect_vocab(train_df["clean_text"].sample(min(20000, len(train_df)), random_state=SEED).tolist())
print(f"Aspect list ({time.time() - t0:.0f}s):", ASPECT_VOCAB)
RESULTS["aspect_vocab"] = ASPECT_VOCAB

In [ ]:
# (a) aspect polarity accuracy on SemEval-2014 test (aspect given)
asp_rows = []
if asp_te is not None:
    ya = asp_te["label"].values
    if ASPECT_HEAD:
        pa, _ = predict_texts(model, asp_te["text"], "asp", pairs=asp_te["aspect"])
        asp_rows.append({"Method": "Aspect head (sentence pair, proposed)", **scores(ya, pa, SENT_LABELS)})
    pf, _ = predict_texts(model, [clause_for(t, a) for t, a in zip(asp_te["text"], asp_te["aspect"])], "sent")
    asp_rows.append({"Method": "Clause + sentiment head (fallback)", **scores(ya, pf, SENT_LABELS)})
    pw, _ = predict_texts(model, asp_te["text"], "sent")
    asp_rows.append({"Method": "Whole-review sentiment (no aspects)", **scores(ya, pw, SENT_LABELS)})
    asp_tab = pd.DataFrame([{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items() if k != "per_class_f1"}
                            for r in asp_rows])
    save_table(asp_tab, "m3_aspect_polarity_semeval")
    RESULTS["aspect_semeval"] = asp_rows

# (b) aspects in the Amazon test reviews
sample = test_df.sample(min(3000, len(test_df)), random_state=SEED).reset_index(drop=True)
t0 = time.time()
sample["aspects"] = extract_aspects(sample["clean_text"].tolist())
pairs = [(i, a) for i, al in enumerate(sample["aspects"]) for a in al]
pol, conf = aspect_polarity([sample["clean_text"][i] for i, _ in pairs], [a for _, a in pairs])
t_asp = time.time() - t0
asp_long = pd.DataFrame({"row": [i for i, _ in pairs], "aspect": [a for _, a in pairs],
                         "polarity": [SENT_LABELS[k] for k in pol], "confidence": conf})
cover = float(np.mean([len(a) > 0 for a in sample["aspects"]]))
RESULTS["aspect_amazon"] = {"reviews": len(sample), "pairs": len(pairs), "coverage_pct": round(cover * 100, 1),
                            "avg_aspects_per_review": round(len(pairs) / len(sample), 2),
                            "seconds": round(t_asp, 1), "pairs_per_sec": round(len(pairs) / max(t_asp, 1e-6), 1)}
print(RESULTS["aspect_amazon"])

top = asp_long.groupby(["aspect", "polarity"]).size().unstack(fill_value=0).reindex(columns=SENT_LABELS, fill_value=0)
top = top.loc[top.sum(1).sort_values(ascending=False).index[:12]]
fig, ax = plt.subplots(figsize=(9, 4.6))
left = np.zeros(len(top))
for lab, col in zip(["Positive", "Neutral", "Negative"], [BLUE, GRAY, RED]):
    ax.barh(top.index, top[lab].values, left=left, color=col, label=lab, height=0.6, edgecolor="white", linewidth=1)
    left += top[lab].values
ax.invert_yaxis(); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_title("Most mentioned aspects and their sentiment (test sample)"); ax.set_xlabel("Mentions"); ax.legend(loc="lower right")
save_fig(fig, "m3_top_aspects")
save_table(top.reset_index(), "m3_top_aspects")

ex = sample[sample["aspects"].str.len() >= 2].head(8).copy()
ex["aspect_sentiment"] = [", ".join(f"{a} → {p}" for a, p in asp_long[asp_long["row"] == i][["aspect", "polarity"]].values)
                          for i in ex.index]
ex_tab = ex[["clean_text", "aspect_sentiment"]].rename(columns={"clean_text": "Review"})
ex_tab["Review"] = ex_tab["Review"].str.slice(0, 160)
save_table(ex_tab, "m3_aspect_examples")

## Module 4 — Explanation and Output  *(report Section 4.5)*
Integrated Gradients scores how much each word pushed the model toward its answer. Then: a hard-case test against the best baseline, the single-sentence result card, bulk-file mode with speed measurement, and the dashboard.

In [ ]:
from captum.attr import LayerIntegratedGradients
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
KW_STOP = set(ENGLISH_STOP_WORDS) - {"not", "no", "never", "nor", "nothing", "cannot", "but", "very", "too"}

def explain(text, task="sent", n_steps=50, top_k=3):
    model.eval()
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    ids, mask = enc["input_ids"].to(DEV), enc["attention_mask"].to(DEV)
    with torch.no_grad():
        probs = model(ids, mask, task).float().softmax(-1)[0]
    pred = int(probs.argmax())
    base = torch.full_like(ids, tok.pad_token_id); base[0, 0] = tok.cls_token_id; base[0, -1] = tok.sep_token_id
    lig = LayerIntegratedGradients(lambda i, m: model(i, m, task), model.encoder.embeddings.word_embeddings)
    attr = lig.attribute(ids, baselines=base, additional_forward_args=(mask,), target=pred,
                         n_steps=n_steps, internal_batch_size=16)
    sc = attr.sum(-1)[0].detach().float().cpu().numpy()
    words, ws = [], []
    for t, s in zip(tok.convert_ids_to_tokens(ids[0].tolist()), sc):
        if t in (tok.cls_token, tok.sep_token, tok.pad_token): continue
        if t.startswith("##") and words: words[-1] += t[2:]; ws[-1] += s
        else: words.append(t); ws.append(s)
    order = np.argsort(ws)[::-1]
    top_words = [words[i] for i in order
                 if ws[i] > 0 and words[i].isalpha() and words[i].lower() not in KW_STOP][:top_k]
    return pred, float(probs[pred]), list(zip(words, map(float, ws))), top_words

def result_card(text):
    s, sc, _, kw = explain(text, "sent")
    e, ec = predict_texts(model, [text], "emo")
    asps = extract_aspects([text])[0]
    ap, _ = aspect_polarity([text] * len(asps), asps) if asps else ([], [])
    return {"review": text, "sentiment": SENT_LABELS[s], "sentiment_confidence": round(sc, 3),
            "emotion": EMO_LABELS[int(e[0])], "emotion_confidence": round(float(ec[0]), 3),
            "aspects": {a: SENT_LABELS[int(p)] for a, p in zip(asps, ap)}, "key_words": kw}

DEMO = "Bad camera quality, but the battery life is great."
card = result_card(DEMO)
print(json.dumps(card, indent=2, ensure_ascii=False))
RESULTS["result_card_demo"] = card

In [ ]:
# Word-importance charts for two examples
examples = [DEMO, "Unreliable connectivity, frequently experiences dropped calls."]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, text in zip(axes, examples):
    pred, conf, pairs_, _ = explain(text)
    w = [p[0] for p in pairs_]; v = np.array([p[1] for p in pairs_])
    ax.bar(range(len(w)), v, color=[BLUE if x > 0 else RED for x in v], width=0.6)
    ax.axhline(0, color="#b9b8b3", lw=0.8)
    ax.set_xticks(range(len(w)), w, rotation=45, ha="right")
    ax.set_title(f"Predicted: {SENT_LABELS[pred]} ({conf:.2f})"); ax.set_ylabel("Word importance")
save_fig(fig, "m4_word_importance")

# Explanations for a few test reviews
rows = []
for t in test_df.sample(8, random_state=SEED)["clean_text"]:
    t = " ".join(t.split()[:60])
    p, c, _, kw = explain(t)
    rows.append({"Review (first 60 words)": t[:160], "Sentiment": SENT_LABELS[p], "Confidence": round(c, 3), "Key words": ", ".join(kw)})
save_table(pd.DataFrame(rows), "m4_explanation_examples")

In [ ]:
# Hard cases: negation, contrast and the base paper's own wrong example
HARD = [("Unreliable connectivity, frequently experiences dropped calls.", 0),
        ("The battery is not good at all.", 0),
        ("Not bad for the price.", 2),
        ("I didn't like the camera.", 0),
        ("I expected it to be terrible, but it works perfectly.", 2),
        ("It is not worth the money.", 0),
        ("Never had a single problem with this charger.", 2),
        ("The case is fine, nothing special.", 1),
        ("Would not recommend this to anyone.", 0),
        ("Works as described, no complaints.", 2),
        ("It stopped charging after two days.", 0),
        ("Not the best case, but it does the job.", 1),
        ("Great, another cable that broke in a week.", 0),
        ("Hardly any battery drain overnight.", 2)]
ht = [h[0] for h in HARD]; hy = np.array([h[1] for h in HARD])
hb = BASE_MODELS[BEST_BASE].predict(vec.transform([base_paper_preprocess(clean_text(t)) for t in ht]))
hp, _ = predict_texts(model, ht, "sent")
hard_tab = pd.DataFrame({"Review": ht, "Expected": [SENT_LABELS[i] for i in hy],
                         f"Base paper ({BEST_BASE})": [SENT_LABELS[i] for i in hb],
                         "Proposed": [SENT_LABELS[i] for i in hp]})
save_table(hard_tab, "m4_hard_cases")
RESULTS["hard_cases"] = {"n": len(HARD), "baseline_correct": int((hb == hy).sum()), "proposed_correct": int((hp == hy).sum()),
                         "baseline_model": BEST_BASE}
print(RESULTS["hard_cases"])

In [ ]:
# Bulk-file mode: read a CSV in chunks, label every row, measure speed
bulk_in = f"{OUT}/bulk_input.csv"
test_df[["review_text", "rating", "product_id", "timestamp"]].to_csv(bulk_in, index=False)
outs, t0 = [], time.time()
for chunk in pd.read_csv(bulk_in, chunksize=CHUNK_SIZE):
    chunk, _ = preprocess(chunk)
    s, sc = predict_texts(model, chunk["clean_text"], "sent")
    e, ec = predict_texts(model, chunk["clean_text"], "emo")
    chunk["sentiment"] = [SENT_LABELS[i] for i in s]; chunk["sentiment_conf"] = np.round(sc, 3)
    chunk["emotion"] = [EMO_LABELS[i] for i in e]; chunk["emotion_conf"] = np.round(ec, 3)
    outs.append(chunk)
labelled = pd.concat(outs, ignore_index=True)
t_model = time.time() - t0
t0 = time.time()
labelled["aspect_list"] = extract_aspects(labelled["clean_text"].tolist())
bp = [(i, a) for i, al in enumerate(labelled["aspect_list"]) for a in al]
bpol, _ = aspect_polarity([labelled["clean_text"][i] for i, _ in bp], [a for _, a in bp])
amap = {}
for (i, a), p in zip(bp, bpol): amap.setdefault(i, []).append(f"{a}:{SENT_LABELS[p]}")
labelled["aspects"] = [", ".join(amap.get(i, [])) for i in range(len(labelled))]
t_aspects = time.time() - t0
labelled.drop(columns=["aspect_list"]).to_csv(f"{OUT}/labelled_dataset.csv", index=False)
RESULTS["bulk"] = {"rows": len(labelled), "sentiment_emotion_seconds": round(t_model, 1),
                   "sentiment_emotion_reviews_per_sec": round(len(labelled) / t_model, 1),
                   "aspect_seconds": round(t_aspects, 1),
                   "total_reviews_per_sec": round(len(labelled) / (t_model + t_aspects), 1), "device": str(DEV)}
print(RESULTS["bulk"])

In [ ]:
# Dashboard for the labelled file
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sc_ = labelled["sentiment"].value_counts().reindex(SENT_LABELS, fill_value=0)
b = axes[0, 0].bar(SENT_LABELS, sc_.values / len(labelled) * 100, color=SENT_COLORS, width=0.6)
bar_labels(axes[0, 0], b, "{:.1f}%"); axes[0, 0].set_title("Predicted sentiment"); axes[0, 0].set_ylabel("% of reviews")
ec_ = labelled["emotion"].value_counts().reindex(EMO_LABELS, fill_value=0)
b = axes[0, 1].bar(EMO_LABELS, ec_.values / len(labelled) * 100, color=BLUE, width=0.6)
bar_labels(axes[0, 1], b, "{:.1f}%"); axes[0, 1].set_title("Predicted emotion"); axes[0, 1].set_ylabel("% of reviews")

al = pd.DataFrame([(a, SENT_LABELS[p]) for (i, a), p in zip(bp, bpol)], columns=["aspect", "polarity"])
if len(al):
    t10 = al.groupby(["aspect", "polarity"]).size().unstack(fill_value=0).reindex(columns=SENT_LABELS, fill_value=0)
    t10 = t10.loc[t10.sum(1).sort_values(ascending=False).index[:10]]
    left = np.zeros(len(t10))
    for lab, col in zip(["Positive", "Neutral", "Negative"], [BLUE, GRAY, RED]):
        axes[1, 0].barh(t10.index, t10[lab].values, left=left, color=col, label=lab, height=0.6, edgecolor="white")
        left += t10[lab].values
    axes[1, 0].invert_yaxis(); axes[1, 0].grid(axis="x"); axes[1, 0].grid(axis="y", visible=False); axes[1, 0].legend(loc="lower right")
axes[1, 0].set_title("Top aspects"); axes[1, 0].set_xlabel("Mentions")

ts = pd.to_datetime(labelled["timestamp"], unit="ms", errors="coerce")
tr = (labelled.assign(q=ts.dt.to_period("Q")).dropna(subset=["q"])
      .groupby("q")["sentiment"].agg(n="size", pos=lambda s: (s == "Positive").mean() * 100))
tr = tr[tr["n"] >= 30].tail(16)
if len(tr):
    axes[1, 1].plot(tr.index.astype(str), tr["pos"].values, "o-", color=BLUE, lw=2, ms=6)
    axes[1, 1].tick_params(axis="x", rotation=45)
else:
    axes[1, 1].text(0.5, 0.5, "Not enough dated reviews per quarter", ha="center", va="center",
                    transform=axes[1, 1].transAxes, color="#52514e")
axes[1, 1].set_title("Positive reviews over time"); axes[1, 1].set_ylabel("% positive"); axes[1, 1].set_xlabel("Quarter")
save_fig(fig, "m4_dashboard")

## Performance and Metrics — existing vs proposed  *(report Section 5)*

In [ ]:
rows = [{"Model": f"{k} (TF-IDF, base paper)", **v} for k, v in BASE_RESULTS.items()]
if RUN_ABLATION:
    rows.append({"Model": "DistilBERT sentiment-only (ablation)", **RESULTS["ablation_sentiment_only"]})
rows.append({"Model": "Multi-task DistilBERT (proposed)", **PROP})
cmp = pd.DataFrame([{"Model": r["Model"], "Accuracy": round(r["accuracy"], 4), "Precision": round(r["precision"], 4),
                     "Recall": round(r["recall"], 4), "Macro-F1": round(r["macro_f1"], 4),
                     "Train time (s)": r["train_seconds"], "Trained on (reviews)": r["trained_on"]} for r in rows])
save_table(cmp, "s5_comparison")
gain = {"accuracy_gain_pts": round((PROP["accuracy"] - BASE_RESULTS[BEST_BASE]["accuracy"]) * 100, 2),
        "macro_f1_gain_pts": round((PROP["macro_f1"] - BASE_RESULTS[BEST_BASE]["macro_f1"]) * 100, 2)}
RESULTS["gain_over_best_baseline"] = gain; print(gain)

short = [r["Model"].split(" (")[0].replace("Multi-task DistilBERT", "Proposed").replace("DistilBERT sentiment-only", "Sentiment-only")
         for r in rows]
x = np.arange(len(rows)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 4.2))
b1 = ax.bar(x - w / 2, cmp["Accuracy"], w, color=BLUE, label="Accuracy")
b2 = ax.bar(x + w / 2, cmp["Macro-F1"], w, color=ORANGE, label="Macro-F1")
bar_labels(ax, b1); bar_labels(ax, b2)
ax.set_xticks(x, short, rotation=15); ax.set_ylim(0, 1.08)
ax.legend(loc="lower right", bbox_to_anchor=(1, 1.0), ncol=2)
ax.set_title("Existing (base paper) vs proposed — test set", loc="left", pad=14); ax.set_ylabel("Score")
save_fig(fig, "s5_accuracy_f1_comparison")

fig, ax = plt.subplots(figsize=(7.5, 3.8)); x = np.arange(3)
b1 = ax.bar(x - w / 2, [BASE_RESULTS[BEST_BASE]["per_class_f1"][l] for l in SENT_LABELS], w, color=GRAY, label=f"Base paper ({BEST_BASE})")
b2 = ax.bar(x + w / 2, [PROP["per_class_f1"][l] for l in SENT_LABELS], w, color=BLUE, label="Proposed")
bar_labels(ax, b1); bar_labels(ax, b2)
ax.set_xticks(x, SENT_LABELS); ax.set_ylim(0, 1.08)
ax.legend(loc="lower right", bbox_to_anchor=(1, 1.0), ncol=2)
ax.set_title("F1 per sentiment class", loc="left", pad=14); ax.set_ylabel("F1")
save_fig(fig, "s5_per_class_f1")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_cm(axes[0], yte, BASE_PREDS[BEST_BASE], SENT_LABELS, f"Base paper ({BEST_BASE})")
plot_cm(axes[1], ys, ps, SENT_LABELS, "Proposed (multi-task)")
save_fig(fig, "s5_confusion_baseline_vs_proposed")

In [ ]:
# Save all numbers and zip the results folder
with open(f"{OUT}/results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
for f_ in ["bulk_input.csv"]:
    if os.path.exists(f"{OUT}/{f_}"): os.remove(f"{OUT}/{f_}")
with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f_ in sorted(os.listdir(OUT)):
        if f_ == "labelled_dataset.csv" and os.path.getsize(f"{OUT}/{f_}") > 20e6: continue
        z.write(f"{OUT}/{f_}", f"results/{f_}")
print("Saved:", sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download("results.zip")
except Exception:
    print("Download results.zip from the file panel on the left.")